In [3]:
import os
import numpy as np, h5py, emcee
from scipy.stats import gaussian_kde
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

H5  = '/scratch/na00078/projects/IPTA_MDC2/h5_files/'
OUT = '/scratch/na00078/projects/IPTA_MDC2/post_processing/'

RUN_D_RAW = H5 + 'G2D1_narrow_UL_4core.h5'                                  # Run D, QuickCW fixed UL, 4 core
LOKI_1PC  = H5 + 'G2D2_fixed_UL_loki_100M_lastTOA_ntol_10_4core.h5'         # Run M 4 core, actual prior 1 percent, name is stale
LOKI_10PC = H5 + 'G2D1_fixed_UL_loki_100M_lastTOA_ntol_10_15_Jul_2026.h5'   # genuine 10 percent, old 1 core
ENT_FILE  = H5 + 'core_single_MDC2_DS1.h5'                                  # Enterprise, not rerun

for p in [RUN_D_RAW, LOKI_1PC, LOKI_10PC, ENT_FILE]:
    print(os.path.exists(p), p.split('/')[-1], flush=True)

Q = 0.95; TARGET = 75.4
NTOL = np.array([0.005,0.01,0.02,0.03,0.05,0.07,0.10,0.15,0.20,0.30,0.50])
megaparsec = 3.086e22; c = 299792458.0; Tsun = 1.327124400e20/c**3
F = np.log10(3.7e-9)

def make_outfile(raw, out, ncol=8, chunk=2_000_000):
    if os.path.exists(out):
        print('exists, skipping', out.split('/')[-1], flush=True); return out
    with h5py.File(raw, 'r') as h:
        d = h['samples_cold']; N = d.shape[1]
        arr = np.empty((N, ncol), dtype=np.float32)
        for a in range(0, N, chunk):
            b = min(a + chunk, N)
            arr[a:b] = d[0, a:b, :ncol]
        pn = h['par_names'][:ncol]
    with h5py.File(out, 'w') as g:
        g.create_dataset('samples_cold', data=arr[None, :, :])
        g.create_dataset('par_names', data=pn)
    print('wrote', out.split('/')[-1], arr.shape, flush=True)
    return out

# Run D must be the UNMASKED chain. G2D1_narrow_UL_4core_outfile.h5 already exists
# but is dL masked (7151 samples), do not use it here.
RUN_D_OUT     = make_outfile(RUN_D_RAW, H5 + 'G2D1_narrow_UL_4core_UNMASKED_outfile.h5')
LOKI_1PC_OUT  = make_outfile(LOKI_1PC,  LOKI_1PC.replace('.h5',  '_outfile.h5'))
LOKI_10PC_OUT = make_outfile(LOKI_10PC, LOKI_10PC.replace('.h5', '_outfile.h5'))

def get_cols(fn, names):
    with h5py.File(fn, 'r') as f:
        pn = [p.decode() if isinstance(p, bytes) else p for p in f['par_names'][:]]
        sc = f['samples_cold'][0, :, :]
        return pn, [sc[:, pn.index(n)].astype(np.float64) for n in names]

# prior width verification. 0_log10_dist is log10 dL, convert to Mpc before taking the span.
# expect about 2 percent of 75.4 for Run M and about 20 percent for the ntol 10 chain
for fn in [LOKI_1PC_OUT, LOKI_10PC_OUT]:
    with h5py.File(fn, 'r') as f:
        pn = [p.decode() if isinstance(p, bytes) else p for p in f['par_names'][:]]
        idx = [k for k, n in enumerate(pn) if 'dl' in n.lower() or 'dist' in n.lower()]
        print(fn.split('/')[-1], 'dL like params:', [pn[k] for k in idx], flush=True)
        for k in idx:
            x = 10.0**f['samples_cold'][0, :, k]
            print('   min %.3f max %.3f Mpc, span %.1f%% of 75.4' % (x.min(), x.max(), 100*(x.max()-x.min())/TARGET), flush=True)

def dl_of(mc, h):
    return 10**(np.log10(2.0)+(5.0/3.0)*(mc+np.log10(Tsun))+(2.0/3.0)*(np.log10(np.pi)+F)-h-np.log10(megaparsec)+np.log10(c))

def tau_of(x):
    th = 10 if len(x) > 5_000_000 else 1
    return th*float(emcee.autocorr.integrated_time(x[::th], quiet=True)[0])

def ul_err(mc_log, n_eff):
    mc = 10**mc_log
    ul = np.quantile(mc, Q)
    f = gaussian_kde(mc if len(mc) < 500_000 else mc[::max(1, len(mc)//500_000)]).evaluate([ul])[0]
    err = np.sqrt(Q*(1-Q))/(f*np.sqrt(n_eff))
    return np.log10(ul), err/(ul*np.log(10))

print('loading unmasked Run D outfile...', flush=True)
_, (mc, h) = get_cols(RUN_D_OUT, ['0_log10_mc', '0_log10_h'])
assert len(mc) == 100_000_000, 'Run D outfile is not the unmasked chain, N=%d' % len(mc)
dl = dl_of(mc, h)
tauD = tau_of(mc); essD = len(mc)/tauD
print('Run D: N=%d tau_mc=%.0f ESS=%.0f' % (len(mc), tauD, essD), flush=True)

rows = []
for nt in NTOL:
    m = np.abs(dl - TARGET) <= nt*TARGET
    ns = int(m.sum())
    neff = min(ns, essD)
    ul, er = ul_err(mc[m], neff)
    rows.append((nt*100, ns, ul, er))
    print(' ntol %.1f%%: n=%d neff=%d UL=%.4f +/- %.4f' % (nt*100, ns, neff, ul, er), flush=True)

def loki_point(fn):
    _, (x,) = get_cols(fn, ['0_log10_mc'])
    tau = tau_of(x); ess = len(x)/tau
    ul, er = ul_err(x, ess)
    print('%s: N=%d tau=%.0f ESS=%.0f UL=%.4f +/- %.4f' % (fn.split('/')[-1], len(x), tau, ess, ul, er), flush=True)
    return len(x), ul, er

nM, ulM, erM = loki_point(LOKI_1PC_OUT)
nC, ulC, erC = loki_point(LOKI_10PC_OUT)

print('loading Enterprise...', flush=True)
with h5py.File(ENT_FILE, 'r') as f:
    pn = [p.decode() if isinstance(p, bytes) else p for p in f['params'][:]]
    ch = f['chain'][...]
    try: burn = f['metadata/burn'][()]
    except Exception: burn = 3000
eMC = ch[burn:, pn.index('log10_mc')]
tauE = tau_of(eMC); essE = len(eMC)/tauE
ulE, erE = ul_err(eMC, essE)
print('ENT: N=%d ESS=%.0f UL=%.4f +/- %.4f' % (len(eMC), essE, ulE, erE), flush=True)

# figure, draft layout: linear x in percent, log y in panel (a) only
plt.rcParams.update({'font.size': 9})
fig, (a, b) = plt.subplots(2, 1, figsize=(3.5, 4.6), sharex=True)
nt = [r[0] for r in rows]; ns = [r[1] for r in rows]; ul = [r[2] for r in rows]; er = [r[3] for r in rows]
a.plot(nt, ns, 'o-', color='#4477AA', ms=4, lw=1.2)
a.scatter([1.0], [nM], marker='D', color='#CC6677', zorder=5, s=28)
a.scatter([10.0], [nC], marker='D', color='#66CCEE', zorder=5, s=28)
a.axhline(len(eMC), color='#228833', ls='--', lw=1.0)
a.set_yscale('log'); a.set_ylabel(r'$N_{\rm surviving}$')
a.text(0.03, 0.86, '(a)', transform=a.transAxes)
b.errorbar(nt, ul, yerr=er, fmt='o-', color='#4477AA', ms=4, lw=1.2, capsize=2)
b.axhline(ulE, color='#228833', ls='--', lw=1.0)
b.fill_between([-2, 54], [ulE-erE]*2, [ulE+erE]*2, color='#228833', alpha=0.18, lw=0)
b.errorbar([1.0], [ulM], yerr=[erM], fmt='D', color='#CC6677', ms=5, capsize=2, zorder=5)
b.errorbar([10.0], [ulC], yerr=[erC], fmt='D', color='#66CCEE', ms=5, capsize=2, zorder=5)
b.set_xlim(-2, 54)
b.set_xlabel(r'$\eta_{\rm tol}$ [%]'); b.set_ylabel(r'$\log_{10}(\mathcal{M}_c/M_\odot)^{95\%}$')
b.text(0.03, 0.86, '(b)', transform=b.transAxes)
plt.tight_layout()
fig.savefig(OUT + 'ntol_sweep_4core.png', dpi=300, bbox_inches='tight')
fig.savefig(OUT + 'ntol_sweep_4core.pdf', bbox_inches='tight')
print('saved ntol_sweep_4core.png/pdf', flush=True)

True G2D1_narrow_UL_4core.h5
True G2D2_fixed_UL_loki_100M_lastTOA_ntol_10_4core.h5
True G2D1_fixed_UL_loki_100M_lastTOA_ntol_10_15_Jul_2026.h5
True core_single_MDC2_DS1.h5
exists, skipping G2D1_narrow_UL_4core_UNMASKED_outfile.h5
exists, skipping G2D2_fixed_UL_loki_100M_lastTOA_ntol_10_4core_outfile.h5
exists, skipping G2D1_fixed_UL_loki_100M_lastTOA_ntol_10_15_Jul_2026_outfile.h5
G2D2_fixed_UL_loki_100M_lastTOA_ntol_10_4core_outfile.h5 dL like params: ['0_log10_dist']
   min 74.646 max 76.154 Mpc, span 2.0% of 75.4
G2D1_fixed_UL_loki_100M_lastTOA_ntol_10_15_Jul_2026_outfile.h5 dL like params: ['0_log10_dist']
   min 67.860 max 82.940 Mpc, span 20.0% of 75.4
loading unmasked Run D outfile...
Run D: N=100000000 tau_mc=9385 ESS=10656
 ntol 0.5%: n=3360 neff=3360 UL=9.7379 +/- 0.0097
 ntol 1.0%: n=7151 neff=7151 UL=9.7928 +/- 0.0078
 ntol 2.0%: n=15297 neff=10655 UL=9.8877 +/- 0.0025
 ntol 3.0%: n=22690 neff=10655 UL=9.8060 +/- 0.0054
 ntol 5.0%: n=37532 neff=10655 UL=9.7823 +/- 0.0035
 n

In [4]:
m1 = np.abs(dl - TARGET) <= 0.01*TARGET
m2 = np.abs(dl - TARGET) <= 0.02*TARGET
new = m2 & ~m1
print('samples entering between 1 and 2 percent:', new.sum())
print('their log10 Mc: min %.3f max %.3f, above 9.85: %d' % (mc[new].min(), mc[new].max(), (mc[new] > 9.85).sum()))
import numpy as np
idx = np.where(new & (mc > 9.85))[0]
print('index range of high Mc entrants:', idx.min() if len(idx) else None, idx.max() if len(idx) else None, 'count', len(idx))

samples entering between 1 and 2 percent: 8146
their log10 Mc: min 7.217 max 9.934, above 9.85: 639
index range of high Mc entrants: 33271162 86907520 count 639


In [ ]:
#!/usr/bin/env python3
import numpy as np, h5py, emcee, json

H5  = '/scratch/na00078/projects/IPTA_MDC2/h5_files/'
OUT = '/scratch/na00078/projects/IPTA_MDC2/post_processing/'
RUNS = {
 'A': ('G2D1_broad_detect_4core.h5',                       34612, 130506),
 'B': ('G2D1_narrow_detect_4core.h5',                      34950, 132314),
 'C': ('G2D1_broad_UL_4core.h5',                           34860, 13663),
 'D': ('G2D1_narrow_UL_4core.h5',                          34947, 7151),
 'E': ('G2D2_broad_detect_tref_4core.h5',                  34091, 192343),
 'F': ('G2D2_narrow_detect_tref_4core.h5',                 33744, 239483),
 'G': ('G2D2_detect_allsky_4core.h5',                      34094, None),
 'L': ('G2D2_broad_UL_loki_100M_lastTOA_4core.h5',         34701, None),
 'M': ('G2D2_fixed_UL_loki_100M_lastTOA_ntol_10_4core.h5', 35622, None),
 'N': ('G2D2_broad_detect_loki_100M_lastTOA_4core.h5',     34058, None),
 'O': ('G2D2_fixed_detect_loki_100M_lastTOA_4core.h5',     34305, None),
}
CHUNK = 2_000_000
res = {}
for r, (fn, tw, nmask) in RUNS.items():
    print('=' * 20, r, fn, flush=True)
    with h5py.File(H5 + fn, 'r') as h:
        pn = [p.decode() if isinstance(p, bytes) else p for p in h['par_names'][:]]
        cols = [i for i, n in enumerate(pn) if n.startswith('0_') or 'gwb' in n.lower()]
        names = [pn[i] for i in cols]
        print(r, 'params:', names, flush=True)
        d = h['samples_cold']; N = d.shape[1]
        arr = np.empty((N, len(cols)), dtype=np.float32)
        for a in range(0, N, CHUNK):
            b = min(a + CHUNK, N)
            arr[a:b] = d[0, a:b, cols]
    taus = {}
    for j, nm in enumerate(names):
        x = arr[::10, j].astype(np.float64)
        if np.all(x == x[0]):
            print(r, nm, 'constant, skipped', flush=True); continue
        tau = 10 * float(emcee.autocorr.integrated_time(x, quiet=True)[0])
        taus[nm] = tau
        print(r, nm, 'tau %.0f' % tau, flush=True)
    tmax = max(taus.values()); ess = N / tmax
    nus = min(ess, nmask) if nmask else ess
    res[r] = dict(file=fn, N=int(N), tau_max=tmax, tau_by_param=taus, ESS=ess,
                  Nmask=nmask, Nusable=nus, Twall=tw, Rpost=nus / tw)
    print('%s: tau_max %.0f  ESS %.0f  Nusable %.0f  Rpost %.2f per s' % (r, tmax, ess, nus, nus / tw), flush=True)
    json.dump(res, open(OUT + 'sec6_ess_4core.json', 'w'), indent=2)
print(json.dumps({k: dict(ESS=round(v['ESS']), Nusable=round(v['Nusable']), Rpost=round(v['Rpost'], 2)) for k, v in res.items()}, indent=2))

==================== A G2D1_broad_detect_4core.h5
A params: ['0_cos_gwtheta', '0_cos_inc', '0_gwphi', '0_log10_fgw', '0_log10_h', '0_log10_mc', '0_phase0', '0_psi', 'gwb_gamma', 'gwb_log10_A']
A 0_cos_gwtheta constant, skipped
A 0_cos_inc tau 15
A 0_gwphi constant, skipped
A 0_log10_fgw tau 7059
A 0_log10_h tau 70
A 0_log10_mc tau 7048
A 0_phase0 tau 14
A 0_psi tau 14
A gwb_gamma constant, skipped
A gwb_log10_A tau 37281
A: tau_max 37281  ESS 2682  Nusable 2682  Rpost 0.08 per s
==================== B G2D1_narrow_detect_4core.h5
B params: ['0_cos_gwtheta', '0_cos_inc', '0_gwphi', '0_log10_fgw', '0_log10_h', '0_log10_mc', '0_phase0', '0_psi', 'gwb_gamma', 'gwb_log10_A']
B 0_cos_gwtheta constant, skipped
B 0_cos_inc tau 11
B 0_gwphi constant, skipped
B 0_log10_fgw constant, skipped
B 0_log10_h tau 15
B 0_log10_mc tau 6346
B 0_phase0 tau 11
B 0_psi tau 11
B gwb_gamma constant, skipped
B gwb_log10_A tau 27661
B: tau_max 27661  ESS 3615  Nusable 3615  Rpost 0.10 per s
==================== C

In [1]:
import os
import numpy as np, h5py, emcee
from scipy.stats import gaussian_kde
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

H5  = '/scratch/na00078/projects/IPTA_MDC2/h5_files/'
OUT = '/scratch/na00078/projects/IPTA_MDC2/post_processing/'

RUN_D_RAW = H5 + 'G2D1_narrow_UL_4core.h5'                                  # Run D, QuickCW fixed UL, 4 core
LOKI_1PC  = H5 + 'G2D2_fixed_UL_loki_100M_lastTOA_ntol_10_4core.h5'         # Run M 4 core, actual prior 1 percent, name is stale
LOKI_10PC = H5 + 'G2D1_fixed_UL_loki_100M_lastTOA_ntol_10_15_Jul_2026.h5'   # genuine 10 percent, old 1 core
ENT_FILE  = H5 + 'core_single_MDC2_DS1.h5'                                  # Enterprise, not rerun

for p in [RUN_D_RAW, LOKI_1PC, LOKI_10PC, ENT_FILE]:
    print(os.path.exists(p), p.split('/')[-1], flush=True)

Q = 0.95; TARGET = 75.4
NTOL = np.array([0.005,0.01,0.02,0.03,0.05,0.07,0.10,0.15,0.20,0.30,0.50])
megaparsec = 3.086e22; c = 299792458.0; Tsun = 1.327124400e20/c**3
F = np.log10(3.7e-9)


def make_outfile(raw, out, ncol=8, chunk=2_000_000):
    if os.path.exists(out):
        print('exists, skipping', out.split('/')[-1], flush=True); return out
    with h5py.File(raw, 'r') as h:
        d = h['samples_cold']; N = d.shape[1]
        arr = np.empty((N, ncol), dtype=np.float32)
        for a in range(0, N, chunk):
            b = min(a + chunk, N)
            arr[a:b] = d[0, a:b, :ncol]
        pn = h['par_names'][:ncol]
    with h5py.File(out, 'w') as g:
        g.create_dataset('samples_cold', data=arr[None, :, :])
        g.create_dataset('par_names', data=pn)
    print('wrote', out.split('/')[-1], arr.shape, flush=True)
    return out


# Run D must be the UNMASKED chain. G2D1_narrow_UL_4core_outfile.h5 already exists
# but is dL masked (7151 samples), do not use it here.
RUN_D_OUT     = make_outfile(RUN_D_RAW, H5 + 'G2D1_narrow_UL_4core_UNMASKED_outfile.h5')
LOKI_1PC_OUT  = make_outfile(LOKI_1PC,  LOKI_1PC.replace('.h5',  '_outfile.h5'))
LOKI_10PC_OUT = make_outfile(LOKI_10PC, LOKI_10PC.replace('.h5', '_outfile.h5'))


def get_cols(fn, names):
    with h5py.File(fn, 'r') as f:
        pn = [p.decode() if isinstance(p, bytes) else p for p in f['par_names'][:]]
        sc = f['samples_cold'][0, :, :]
        return pn, [sc[:, pn.index(n)].astype(np.float64) for n in names]


# prior width verification. 0_log10_dist is log10 dL, convert to Mpc before taking the span.
# expect about 2 percent of 75.4 for Run M and about 20 percent for the ntol 10 chain
for fn in [LOKI_1PC_OUT, LOKI_10PC_OUT]:
    with h5py.File(fn, 'r') as f:
        pn = [p.decode() if isinstance(p, bytes) else p for p in f['par_names'][:]]
        idx = [k for k, n in enumerate(pn) if 'dl' in n.lower() or 'dist' in n.lower()]
        print(fn.split('/')[-1], 'dL like params:', [pn[k] for k in idx], flush=True)
        for k in idx:
            x = 10.0**f['samples_cold'][0, :, k]
            print('   min %.3f max %.3f Mpc, span %.1f%% of 75.4' % (x.min(), x.max(), 100*(x.max()-x.min())/TARGET), flush=True)


def dl_of(mc, h):
    return 10**(np.log10(2.0)+(5.0/3.0)*(mc+np.log10(Tsun))+(2.0/3.0)*(np.log10(np.pi)+F)-h-np.log10(megaparsec)+np.log10(c))


def tau_of(x):
    """Integrated autocorrelation time, in units of the index of the array handed in.

    For a masked subset this is the correlation time measured along the surviving
    samples in chain order, which is what sets how many independent draws that
    subset actually contains.
    """
    x = np.asarray(x, dtype=np.float64)
    n = x.size
    if n < 100:
        return float(max(n, 1))
    th = 10 if n > 5_000_000 else 1
    try:
        t = th*float(emcee.autocorr.integrated_time(x[::th], quiet=True)[0])
    except Exception:
        t = np.nan
    if not np.isfinite(t) or t < 1.0:
        t = 1.0
    return t


def ul_err_log(mc_log, n_eff):
    """95 percent upper limit on log10 Mc and its standard error.

    Asymptotic quantile standard error, sigma = sqrt(q(1-q)/n_eff) / f(x_q),
    evaluated directly in log10 Mc so no Jacobian conversion is needed and the
    KDE is not asked to resolve a density spanning several decades in linear Mc.
    """
    x = np.asarray(mc_log, dtype=np.float64)
    ul = float(np.quantile(x, Q))
    xs = x if x.size < 500_000 else x[::max(1, x.size//500_000)]
    if xs.size < 20 or np.ptp(xs) == 0.0:
        return ul, np.nan
    try:
        f = float(gaussian_kde(xs).evaluate([ul])[0])
    except Exception:
        return ul, np.nan
    if not np.isfinite(f) or f <= 0.0 or n_eff <= 0:
        return ul, np.nan
    return ul, float(np.sqrt(Q*(1.0-Q))/(f*np.sqrt(n_eff)))


print('loading unmasked Run D outfile...', flush=True)
_, (mc, h) = get_cols(RUN_D_OUT, ['0_log10_mc', '0_log10_h'])
assert len(mc) == 100_000_000, 'Run D outfile is not the unmasked chain, N=%d' % len(mc)
dl = dl_of(mc, h)

tauD = tau_of(mc); essD = len(mc)/tauD
print('Run D unmasked: N=%d tau_mc=%.0f ESS=%.0f' % (len(mc), tauD, essD), flush=True)

# ---------------------------------------------------------------------------
# eta_tol sweep. n_eff is now estimated from the masked subset itself, per point,
# instead of reusing the full chain ESS for every point. The full chain ESS is
# still applied as a ceiling, since a subset cannot carry more independent
# information than the chain it was drawn from.
# ---------------------------------------------------------------------------
rows = []
for nt in NTOL:
    m = np.abs(dl - TARGET) <= nt*TARGET
    ns = int(m.sum())
    sub = mc[m]
    tau_s = tau_of(sub)
    ess_s = ns/tau_s
    neff = min(ess_s, essD)
    capped = ess_s > essD
    ul, er = ul_err_log(sub, neff)
    rows.append((nt*100, ns, tau_s, ess_s, neff, ul, er, capped))
    print(' ntol %5.1f%%  n=%8d  tau_sub=%8.1f  ess_sub=%9.1f  neff=%9.1f%s  UL=%.4f +/- %.4f'
          % (nt*100, ns, tau_s, ess_s, neff, '  [capped at chain ESS]' if capped else '', ul, er), flush=True)


def loki_point(fn):
    _, (x,) = get_cols(fn, ['0_log10_mc'])
    tau = tau_of(x); ess = len(x)/tau
    ul, er = ul_err_log(x, ess)
    print('%s: N=%d tau=%.0f ESS=%.0f UL=%.4f +/- %.4f' % (fn.split('/')[-1], len(x), tau, ess, ul, er), flush=True)
    return len(x), ul, er


nM, ulM, erM = loki_point(LOKI_1PC_OUT)
nC, ulC, erC = loki_point(LOKI_10PC_OUT)

print('loading Enterprise...', flush=True)
with h5py.File(ENT_FILE, 'r') as f:
    pn = [p.decode() if isinstance(p, bytes) else p for p in f['params'][:]]
    ch = f['chain'][...]
    try: burn = f['metadata/burn'][()]
    except Exception: burn = 3000
eMC = ch[burn:, pn.index('log10_mc')]
tauE = tau_of(eMC); essE = len(eMC)/tauE
ulE, erE = ul_err_log(eMC, essE)
print('ENT: N=%d ESS=%.0f UL=%.4f +/- %.4f' % (len(eMC), essE, ulE, erE), flush=True)

# ---------------------------------------------------------------------------
# results table, printed in this cell and written to disk
# ---------------------------------------------------------------------------
hdr = ('%8s %10s %10s %11s %11s %10s %9s' %
       ('ntol[%]', 'N_surv', 'tau_sub', 'ESS_sub', 'n_eff', 'log10 UL', 'err'))
print('\n' + '=' * len(hdr), flush=True)
print('eta_tol sweep, Run D (QuickCW, GWB only, fixed fGW), 4 core, unmasked N = 1e8', flush=True)
print('=' * len(hdr), flush=True)
print(hdr, flush=True)
print('-' * len(hdr), flush=True)
for r in rows:
    print('%8.1f %10d %10.1f %11.1f %11.1f %10.4f %9.4f' % (r[0], r[1], r[2], r[3], r[4], r[5], r[6]), flush=True)
print('-' * len(hdr), flush=True)
print('%-28s %10.4f +/- %.4f  (N=%d, ESS=%.0f)' % ('Enterprise', ulE, erE, len(eMC), essE), flush=True)
print('%-28s %10.4f +/- %.4f  (N=%d)' % ('QuickCW-dL, 1 percent prior', ulM, erM, nM), flush=True)
print('%-28s %10.4f +/- %.4f  (N=%d)' % ('QuickCW-dL, 10 percent prior', ulC, erC, nC), flush=True)
print('=' * len(hdr) + '\n', flush=True)

csv_path = OUT + 'ntol_sweep_4core_results.csv'
with open(csv_path, 'w') as fh:
    fh.write('ntol_percent,N_surviving,tau_sub,ESS_sub,n_eff,log10_Mc_UL95,err\n')
    for r in rows:
        fh.write('%.3f,%d,%.4f,%.4f,%.4f,%.6f,%.6f\n' % (r[0], r[1], r[2], r[3], r[4], r[5], r[6]))
    fh.write('# enterprise,%d,%.4f,%.4f,%.4f,%.6f,%.6f\n' % (len(eMC), tauE, essE, essE, ulE, erE))
    fh.write('# loki_1pc,%d,,,,%.6f,%.6f\n' % (nM, ulM, erM))
    fh.write('# loki_10pc,%d,,,,%.6f,%.6f\n' % (nC, ulC, erC))
print('wrote', csv_path, flush=True)

# ---------------------------------------------------------------------------
# figure. panel (a) unchanged. panel (b) same points, error bars now per point.
# ---------------------------------------------------------------------------
plt.rcParams.update({'font.size': 9})
fig, (a, b) = plt.subplots(2, 1, figsize=(3.5, 4.6), sharex=True)

nt = [r[0] for r in rows]; ns = [r[1] for r in rows]; ul = [r[5] for r in rows]; er = [r[6] for r in rows]

a.plot(nt, ns, 'o-', color='#4477AA', ms=4, lw=1.2)
a.scatter([1.0], [nM], marker='D', color='#CC6677', zorder=5, s=28)
a.scatter([10.0], [nC], marker='D', color='#66CCEE', zorder=5, s=28)
a.axhline(len(eMC), color='#228833', ls='--', lw=1.0)
a.set_yscale('log'); a.set_ylabel(r'$N_{\rm surviving}$')
a.text(0.03, 0.86, '(a)', transform=a.transAxes)

b.errorbar(nt, ul, yerr=er, fmt='o-', color='#4477AA', ms=4, lw=1.2, capsize=2)
b.axhline(ulE, color='#228833', ls='--', lw=1.0)
b.fill_between([-2, 54], [ulE-erE]*2, [ulE+erE]*2, color='#228833', alpha=0.18, lw=0)
b.errorbar([1.0], [ulM], yerr=[erM], fmt='D', color='#CC6677', ms=5, capsize=2, zorder=5)
b.errorbar([10.0], [ulC], yerr=[erC], fmt='D', color='#66CCEE', ms=5, capsize=2, zorder=5)
b.set_xlim(-2, 54)
b.set_xlabel(r'$\eta_{\rm tol}$ [%]'); b.set_ylabel(r'$\log_{10}(\mathcal{M}_c/M_\odot)^{95\%}$')
b.text(0.03, 0.86, '(b)', transform=b.transAxes)

plt.tight_layout()
fig.savefig(OUT + 'ntol_sweep_4core.png', dpi=300, bbox_inches='tight')
fig.savefig(OUT + 'ntol_sweep_4core.pdf', bbox_inches='tight')
print('saved ntol_sweep_4core.png/pdf', flush=True)

True G2D1_narrow_UL_4core.h5
True G2D2_fixed_UL_loki_100M_lastTOA_ntol_10_4core.h5
True G2D1_fixed_UL_loki_100M_lastTOA_ntol_10_15_Jul_2026.h5
True core_single_MDC2_DS1.h5
exists, skipping G2D1_narrow_UL_4core_UNMASKED_outfile.h5
exists, skipping G2D2_fixed_UL_loki_100M_lastTOA_ntol_10_4core_outfile.h5
exists, skipping G2D1_fixed_UL_loki_100M_lastTOA_ntol_10_15_Jul_2026_outfile.h5
G2D2_fixed_UL_loki_100M_lastTOA_ntol_10_4core_outfile.h5 dL like params: ['0_log10_dist']
   min 74.646 max 76.154 Mpc, span 2.0% of 75.4
G2D1_fixed_UL_loki_100M_lastTOA_ntol_10_15_Jul_2026_outfile.h5 dL like params: ['0_log10_dist']
   min 67.860 max 82.940 Mpc, span 20.0% of 75.4
loading unmasked Run D outfile...
Run D unmasked: N=100000000 tau_mc=9385 ESS=10656
 ntol   0.5%  n=    3360  tau_sub=     9.2  ess_sub=    363.4  neff=    363.4  UL=9.7379 +/- 0.0230
 ntol   1.0%  n=    7151  tau_sub=    29.3  ess_sub=    244.2  neff=    244.2  UL=9.7928 +/- 0.0353
 ntol   2.0%  n=   15297  tau_sub=    70.0  ess_s